In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from scipy.stats import ttest_rel

ridge = Ridge()

phys_cv = cross_val_score(
    ridge, X_train_phys_sample, y_train_esm,
    cv=5, scoring="neg_mean_absolute_error"
)

esm_cv = cross_val_score(
    ridge, X_train_esm, y_train_esm,
    cv=5, scoring="neg_mean_absolute_error"
)

hybrid_cv = cross_val_score(
    ridge, X_train_hybrid, y_train_esm,
    cv=5, scoring="neg_mean_absolute_error"
)

print("Hybrid vs Phys:", ttest_rel(-hybrid_cv, -phys_cv)[1])
print("Hybrid vs ESM:", ttest_rel(-hybrid_cv, -esm_cv)[1])

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np
import pandas as pd

# baseline

phys_model = Ridge().fit(X_train_phys_sample, y_train_esm)
phys_preds = phys_model.predict(X_test_phys_sample)

esm_model = Ridge().fit(X_train_esm, y_train_esm)
esm_preds = esm_model.predict(X_test_esm)

# hybrid

hybrid_model = Ridge().fit(X_train_hybrid, y_train_esm)
hybrid_preds = hybrid_model.predict(X_test_hybrid)

# ablation (no embeddings)

ablated_model = Ridge().fit(X_train_phys_sample, y_train_esm)
ablated_preds = ablated_model.predict(X_test_phys_sample)

# results

ablation_results = pd.DataFrame([
    {"Model": "Physicochemical", "MAE": mean_absolute_error(y_test_esm, phys_preds)},
    {"Model": "ESM Only", "MAE": mean_absolute_error(y_test_esm, esm_preds)},
    {"Model": "Hybrid", "MAE": mean_absolute_error(y_test_esm, hybrid_preds)},
    {"Model": "Ablated (No ESM)", "MAE": mean_absolute_error(y_test_esm, ablated_preds)}
])

print(ablation_results)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np
import pandas as pd

# shuffle labels

y_train_shuffled = np.random.permutation(y_train_esm)

# models trained on noise

phys_nc = Ridge().fit(X_train_phys_sample, y_train_shuffled)
esm_nc = Ridge().fit(X_train_esm, y_train_shuffled)
hybrid_nc = Ridge().fit(X_train_hybrid, y_train_shuffled)

# predictions
phys_nc_preds = phys_nc.predict(X_test_phys_sample)
esm_nc_preds = esm_nc.predict(X_test_esm)
hybrid_nc_preds = hybrid_nc.predict(X_test_hybrid)

# results

negative_control_results = pd.DataFrame([
    {
        "Model": "Physicochemical (Shuffled)",
        "MAE": mean_absolute_error(y_test_esm, phys_nc_preds),
        "R2": r2_score(y_test_esm, phys_nc_preds)
    },
    {
        "Model": "ESM (Shuffled)",
        "MAE": mean_absolute_error(y_test_esm, esm_nc_preds),
        "R2": r2_score(y_test_esm, esm_nc_preds)
    },
    {
        "Model": "Hybrid (Shuffled)",
        "MAE": mean_absolute_error(y_test_esm, hybrid_nc_preds),
        "R2": r2_score(y_test_esm, hybrid_nc_preds)
    }
])

print("\nNEGATIVE CONTROL RESULTS")
print(negative_control_results)